# SME Legal QA — 2K Sample Benchmark (Colab A100)

**Target**: Process 2000 questions in ~1-2 hours on Colab Pro A100.

**Optimizations**:
- GPU-efficient batch processing (FAISS GPU, reranker batch=16)
- Streaming writes (flush every 100 questions → no OOM)
- Minimal logging overhead
- Pre-load all indices once

**Output**: `results_2k.jsonl` with lane, F2 per question, and timing stats.

In [ ]:
# ============================================================================
# 1. Environment setup (Colab A100)
# ============================================================================
!nvidia-smi
!pip install -q faiss-gpu pyarrow pyyaml regex tqdm networkx FlagEmbedding transformers accelerate

# Mount Google Drive (assumes your data artifacts are in Drive)
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/Road2AI_ApplePie/src')

In [ ]:
# ============================================================================
# 2. Load artifacts (once)
# ============================================================================
import json
import pickle
import sqlite3
from pathlib import Path

import faiss
import networkx as nx
import pandas as pd
import yaml
from tqdm.auto import tqdm

# Adjust paths to your Drive structure
BASE = Path('/content/drive/MyDrive/Road2AI_ApplePie')
DATA = BASE / 'data/stage6_data'

# Config
with open(BASE / 'config/default.yaml') as f:
    cfg = yaml.safe_load(f)

# BM25 index (SQLite FTS5)
bm25_conn = sqlite3.connect(DATA / 'chunk_store.sqlite')
print(f"✓ BM25 index: {bm25_conn.execute('SELECT COUNT(*) FROM chunk_fts').fetchone()[0]} chunks")

# Chunk metadata (row_idx → law_id, dieu_so, etc.)
meta_df = pd.read_parquet(DATA / 'chunk_meta_slim.parquet')
print(f"✓ Metadata: {len(meta_df)} rows")

# FAISS index (GPU)
faiss_cpu = faiss.read_index(str(DATA / 'faiss_index__BAAI_bge-m3.index'))
res = faiss.StandardGpuResources()
faiss_gpu = faiss.index_cpu_to_gpu(res, 0, faiss_cpu)
print(f"✓ FAISS (GPU): {faiss_gpu.ntotal} vectors")

# NetworkX graph
with open(BASE / 'data/kg.gpickle', 'rb') as f:
    G = pickle.load(f)
print(f"✓ Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# row_to_uid mapping for GraphExpander
row_to_uid = dict(zip(meta_df['row_idx'], meta_df['doc_uid']))

# ---- PERF-CRITICAL: prebuild O(1) lookup dicts ----
# Repeated meta_df[meta_df.row_idx==x] / meta_df.iloc[x] scans during 2000×70
# lookups would add HOURS. Build dicts once.
#
# 1) FAISS position → metadata. The Stage 6 invariant is that FAISS vector
#    position i corresponds to meta_df row i (row_idx == i, contiguous). We
#    map by positional order to be robust, keyed by FAISS idx.
meta_records = meta_df.to_dict('records')
pos_to_meta = {i: rec for i, rec in enumerate(meta_records)}
row_to_meta = {int(rec['row_idx']): rec for rec in meta_records}

# 2) row_idx → chunk_text, pulled ONCE from the SQLite chunk store (text lives
#    there, not in the slim parquet). One bulk query → dict.
text_by_row = {}
try:
    cur = bm25_conn.execute('SELECT row_idx, text FROM chunks')
    for ridx, txt in cur.fetchall():
        text_by_row[int(ridx)] = txt or ''
    print(f"✓ Text cache: {len(text_by_row)} chunks (from chunks table)")
except Exception as e:
    # Fallback: text column may live in the FTS table or the parquet.
    print(f"⚠ chunks.text query failed ({e}); trying parquet column")
    if 'chunk_text' in meta_df.columns:
        text_by_row = {int(r['row_idx']): (r.get('chunk_text') or '')
                       for r in meta_records}
        print(f"✓ Text cache from parquet: {len(text_by_row)} chunks")

del faiss_cpu  # free CPU copy

In [ ]:
# ============================================================================
# 3. Load models (embedding + reranker, GPU batch mode)
# ============================================================================
from FlagEmbedding import BGEM3FlagModel, FlagReranker

# Query encoder (for FAISS dense search)
embed_model = BGEM3FlagModel(
    'BAAI/bge-m3',
    use_fp16=True,
    device='cuda:0'
)
print("✓ Embedding model loaded")

# Reranker (batch_size=16 for A100)
reranker = FlagReranker(
    'BAAI/bge-reranker-v2-m3',
    use_fp16=True,
    device='cuda:0'
)
print("✓ Reranker loaded")

In [ ]:
# ============================================================================
# 4. Build the retrieval pipeline
# ============================================================================
from retrieval.router import Router, RouterConfig
from retrieval.decomposer import Decomposer, DecomposerConfig
from retrieval.graph_expand import GraphExpander
from retrieval.final_selector import FinalSelector, SelectionConfig
from retrieval.retrieval_pipeline import (
    RetrievalPipeline,
    RetrievalConfig,
)
from retrieval.bm25_index import tokenize_query
import numpy as np

# BM25 search callable
def bm25_search(query: str, top_k: int):
    tokens = tokenize_query(query)
    if not tokens:
        return []
    fts_query = ' OR '.join(f'"{t}"' for t in tokens[:20])
    sql = f"""
        SELECT row_idx, rank, law_id, ten_van_ban, dieu_so
        FROM chunk_fts
        WHERE chunk_fts MATCH ?
        ORDER BY rank
        LIMIT ?
    """
    rows = bm25_conn.execute(sql, (fts_query, top_k)).fetchall()
    return [
        {
            'row_idx': r[0],
            'score': -float(r[1]),  # rank is negative; negate for descending
            'law_id': r[2],
            'ten_van_ban': r[3],
            'dieu_so': r[4],
        }
        for r in rows
    ]

# FAISS dense search callable (O(1) metadata lookup via pos_to_meta)
def dense_search(query: str, top_k: int):
    emb = embed_model.encode([query], return_dense=True, return_sparse=False)['dense_vecs']
    emb = emb.astype('float32')
    faiss.normalize_L2(emb)
    D, I = faiss_gpu.search(emb, top_k)
    hits = []
    for dist, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        rec = pos_to_meta.get(int(idx))
        if rec is None:
            continue
        hits.append({
            'row_idx': int(rec['row_idx']),
            'score': float(dist),
            'law_id': rec['law_id'],
            'ten_van_ban': rec['ten_van_ban'],
            'dieu_so': rec['dieu_so'],
        })
    return hits

# Text provider (O(1) dict lookup from the prebuilt text cache)
def text_provider(row_idxs):
    return {int(r): text_by_row.get(int(r), '') for r in row_idxs}

# Reranker callable (batch mode). compute_score is itself batched on GPU; we
# hand it ALL pairs at once and let batch_size chunk them — far faster than
# per-pair calls. 64 is comfortable for an A100's 40-80GB with bge-reranker-v2-m3.
def rerank_fn(query: str, passages):
    if not passages:
        return []
    pairs = [[query, p] for p in passages]
    scores = reranker.compute_score(pairs, batch_size=64, normalize=True)
    if isinstance(scores, (int, float)):
        return [float(scores)]
    return [float(s) for s in scores]

# Graph expander
graph_expander = GraphExpander(G, row_to_uid)

# Build pipeline
router = Router(config=RouterConfig(use_llm=False))
decomposer = Decomposer(config=DecomposerConfig(use_llm=False))
selector = FinalSelector(config=SelectionConfig(drop_provincial=False))

pipeline = RetrievalPipeline(
    router=router,
    decomposer=decomposer,
    lexical_search=bm25_search,
    dense_search=dense_search,
    reranker=rerank_fn,
    text_provider=text_provider,
    graph_expander=graph_expander,
    final_selector=selector,
    cfg=RetrievalConfig(
        top_bm25=80,
        top_dense=80,
        rrf_k=60,
        fused_pool_size=120,
        use_dense=True,
        rerank_input_size=70,
        rerank_text_truncate=512,
        graph_expand_seeds=8,
        graph_expanded_top=60,
    ),
)

print("✓ Pipeline ready")

In [ ]:
# ============================================================================
# 5. Load 2000 questions (adjust path to your test set)
# ============================================================================
import random

# Example: load from a JSONL file with {question_id, question, ground_truth}
# If you don't have one, generate synthetic or sample from train set.
test_path = BASE / 'data/test_2k.jsonl'

if not test_path.exists():
    print("⚠ test_2k.jsonl not found. Creating 2000 dummy questions for demo.")
    questions = [
        {
            'question_id': f'q{i}',
            'question': f'Điều kiện hưởng ưu đãi thuế thu nhập doanh nghiệp là gì? (sample {i})',
            'ground_truth': [],
        }
        for i in range(2000)
    ]
else:
    with open(test_path) as f:
        questions = [json.loads(line) for line in f]
    questions = questions[:2000]

print(f"✓ Loaded {len(questions)} questions")

In [ ]:
# ============================================================================
# 6. Process 2000 questions with streaming output (1-2 hours target)
# ============================================================================
import time

output_path = BASE / 'results_2k.jsonl'
output_file = open(output_path, 'w', encoding='utf-8')

start_time = time.time()
results = []

for i, item in enumerate(tqdm(questions, desc='Processing')):
    qid = item['question_id']
    query = item['question']
    
    try:
        t0 = time.time()
        result = pipeline.retrieve(query)
        elapsed = time.time() - t0
        
        record = {
            'question_id': qid,
            'question': query,
            'lane': result.routing.lane,
            'predicted_articles': result.relevant_articles(),
            'ground_truth': item.get('ground_truth', []),
            'time_sec': round(elapsed, 3),
            'stage_stats': result.stage_stats,
        }
        output_file.write(json.dumps(record, ensure_ascii=False) + '\n')
        
        # Flush every 100 to avoid memory buildup
        if (i + 1) % 100 == 0:
            output_file.flush()
            elapsed_total = time.time() - start_time
            avg_per_q = elapsed_total / (i + 1)
            eta_sec = avg_per_q * (len(questions) - i - 1)
            print(f"[{i+1}/{len(questions)}] {avg_per_q:.2f}s/q | ETA: {eta_sec/60:.1f}min")
    
    except Exception as e:
        print(f"Error on {qid}: {e}")
        output_file.write(json.dumps({'question_id': qid, 'error': str(e)}) + '\n')

output_file.close()
total_time = time.time() - start_time
print(f"\n✓ Done: {len(questions)} questions in {total_time/60:.1f} minutes")
print(f"  Average: {total_time/len(questions):.2f}s/question")
print(f"  Output: {output_path}")

In [ ]:
# ============================================================================
# 7. Compute F2 macro (if ground truth is available)
# ============================================================================
def canonical_dieu(s):
    import re
    s = str(s).strip()
    m = re.match(r'^(Điều)\s*(.+)$', s, re.UNICODE)
    if m:
        return f"{m.group(1)} {m.group(2).strip()}"
    m2 = re.search(r'Điều\s*(\d+[a-zA-Z]*(?:[.\-]\d+)?)', s, re.UNICODE)
    if m2:
        return f"Điều {m2.group(1)}"
    return s

def extract_dieu_set(articles):
    """articles: list of 'law|ten|Điều X' strings."""
    dieus = set()
    for art in articles:
        parts = art.split('|')
        if len(parts) >= 3:
            dieus.add(canonical_dieu(parts[2]))
    return dieus

def f2_score(pred_set, gold_set):
    if not gold_set:
        return 0.0 if pred_set else 1.0
    if not pred_set:
        return 0.0
    tp = len(pred_set & gold_set)
    prec = tp / len(pred_set)
    rec = tp / len(gold_set)
    if prec + rec == 0:
        return 0.0
    return 5 * prec * rec / (4 * prec + rec)

# Load results
with open(output_path) as f:
    results = [json.loads(line) for line in f if 'error' not in json.loads(line)]

f2_scores = []
for r in results:
    pred = extract_dieu_set(r['predicted_articles'])
    gold = extract_dieu_set(r['ground_truth'])
    f2_scores.append(f2_score(pred, gold))

f2_macro = sum(f2_scores) / len(f2_scores) if f2_scores else 0.0
print(f"\nF2 macro (2000 questions): {f2_macro:.4f}")

# Lane distribution
from collections import Counter
lanes = Counter(r['lane'] for r in results)
print(f"\nLane distribution:")
for lane, count in lanes.most_common():
    print(f"  {lane}: {count}")

In [ ]:
# ============================================================================
# 8. Download results to Drive (backup)
# ============================================================================
print(f"Results saved to: {output_path}")
print(f"Copy to your local machine or keep in Drive.")